# 75. Privacy Protection

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/09-adversarial/75_privacy_protection.ipynb)

**Category:** Adversarial & Safety  **Technique #:** 75  **Difficulty:** Advanced

## Description

Privacy protection in AI systems involves safeguarding sensitive personal information from unauthorized access, exposure, or misuse. This technique covers strategies to prevent data leaks, protect user privacy, and ensure compliance with regulations like GDPR, CCPA, and HIPAA.

**When to use:**
- Processing user data in AI applications
- Building systems that handle PII (Personally Identifiable Information)
- Working in regulated industries (healthcare, finance, legal)
- Implementing data minimization practices
- Ensuring GDPR/CCPA compliance

## How It Works

```
┌─────────────────────────────────────────────────────────────┐
│                    PRIVACY PROTECTION                       │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  Input ──► [PII Detection] ──► [Classification] ──► [Action]│
│               │                     │               │       │
│               ▼                     ▼               ▼       │
│          Pattern Match         Sensitivity       Redact/  │
│          ML Detection          Level             Block/   │
│                                                  Encrypt    │
│                                                             │
│  PII Categories:                                            │
│  ┌──────────┐ ┌──────────┐ ┌──────────┐ ┌──────────┐       │
│  │ Personal │ │ Contact  │ │ Financial│ │ Medical  │       │
│  │ Identity │ │ Info     │ │ Data     │ │ Records  │       │
│  └──────────┘ └──────────┘ └──────────┘ └──────────┘       │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

**Protection Strategies:**
1. **Data Minimization** - Collect only necessary data
2. **PII Detection** - Identify sensitive information
3. **Anonymization** - Remove or mask identifiers
4. **Encryption** - Protect data at rest and in transit
5. **Access Control** - Limit who can access data
6. **Audit Logging** - Track data access and usage

## Setup

In [ ]:
# Install required packages
!pip install -q openai presidio-analyzer presidio-anonymizer

import openai
import re
import json
import hashlib
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field
from enum import Enum
from getpass import getpass

# Set up OpenAI API key
openai.api_key = getpass("Enter your OpenAI API key: ")

print("✅ Setup complete!")

## Basic Example: PII Detection and Protection

In [ ]:
class PIICategory(Enum):
    NONE = "none"
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"
    CRITICAL = "critical"

@dataclass
class PIIDetectionResult:
    original_text: str
    sanitized_text: str
    pii_found: List[Dict]
    risk_level: PIICategory
    action_taken: str

class PrivacyProtector:
    PII_PATTERNS = {
        'email': {'pattern': r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', 'sensitivity': PIICategory.MEDIUM},
        'phone': {'pattern': r'\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b', 'sensitivity': PIICategory.MEDIUM},
        'ssn': {'pattern': r'\b\d{3}-\d{2}-\d{4}\b', 'sensitivity': PIICategory.CRITICAL},
        'credit_card': {'pattern': r'\b\d{4}[-\s]?\d{4}[-\s]?\d{4}[-\s]?\d{4}\b', 'sensitivity': PIICategory.CRITICAL},
        'ip_address': {'pattern': r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b', 'sensitivity': PIICategory.LOW}
    }
    
    def __init__(self, mask_char: str = 'X'):
        self.mask_char = mask_char
        self.compiled_patterns = {}
        for pii_type, config in self.PII_PATTERNS.items():
            self.compiled_patterns[pii_type] = {
                'regex': re.compile(config['pattern']),
                'sensitivity': config['sensitivity']
            }
    
    def detect_pii(self, text: str) -> List[Dict]:
        pii_found = []
        for pii_type, config in self.compiled_patterns.items():
            matches = config['regex'].finditer(text)
            for match in matches:
                pii_found.append({'type': pii_type, 'value': match.group(), 'start': match.start(), 'end': match.end(), 'sensitivity': config['sensitivity']})
        return pii_found
    
    def mask_pii(self, text: str, pii_items: List[Dict]) -> str:
        sorted_items = sorted(pii_items, key=lambda x: x['start'], reverse=True)
        masked_text = text
        for item in sorted_items:
            value = item['value']
            if item['type'] in ['email', 'phone'] and len(value) > 6:
                masked = value[:2] + self.mask_char * (len(value) - 4) + value[-2:]
            else:
                masked = self.mask_char * len(value)
            masked_text = masked_text[:item['start']] + masked + masked_text[item['end']:]
        return masked_text
    
    def protect(self, text: str, action: str = 'mask') -> PIIDetectionResult:
        pii_found = self.detect_pii(text)
        if not pii_found:
            return PIIDetectionResult(original_text=text, sanitized_text=text, pii_found=[], risk_level=PIICategory.NONE, action_taken='none')
        max_sensitivity = max(pii['sensitivity'] for pii in pii_found)
        if action == 'mask':
            sanitized = self.mask_pii(text, pii_found)
        elif action == 'hash':
            sanitized = text
            for pii in sorted(pii_found, key=lambda x: x['start'], reverse=True):
                hash_val = hashlib.md5(pii['value'].encode()).hexdigest()[:8]
                sanitized = sanitized[:pii['start']] + f'[HASH:{hash_val}]' + sanitized[pii['end']:]
        else:
            sanitized = text
        return PIIDetectionResult(original_text=text, sanitized_text=sanitized, pii_found=pii_found, risk_level=max_sensitivity, action_taken=action)

protector = PrivacyProtector()
test_texts = [
    'Contact me at john.doe@email.com or call 555-123-4567',
    'My SSN is 123-45-6789 and credit card is 1234-5678-9012-3456',
    'The meeting is at 3 PM tomorrow',
    'Server IP: 192.168.1.1'
]
print('=== PII Detection and Protection ===\n')
for text in test_texts:
    result = protector.protect(text, action='mask')
    print(f'Original: {text}')
    print(f'Sanitized: {result.sanitized_text}')
    print(f'Risk Level: {result.risk_level.value}')
    print()

## Real-World Example: Privacy-Preserving AI Assistant

In [ ]:
class PrivacyPreservingAssistant:
    SYSTEM_PROMPT = '''You are a helpful assistant with strict privacy guidelines.
NEVER store PII between conversations. If PII is shared, confirm it won't be stored.'''
    
    def __init__(self):
        self.protector = PrivacyProtector()
        self.privacy_log = []
    
    def process_message(self, user_message: str) -> Dict:
        privacy_check = self.protector.protect(user_message, action='mask')
        if privacy_check.pii_found:
            self.privacy_log.append({'pii_types': [pii['type'] for pii in privacy_check.pii_found], 'risk_level': privacy_check.risk_level.value})
        safe_message = privacy_check.sanitized_text
        try:
            response = openai.chat.completions.create(
                model='gpt-3.5-turbo',
                messages=[{'role': 'system', 'content': self.SYSTEM_PROMPT}, {'role': 'user', 'content': safe_message}],
                temperature=0.5, max_tokens=300
            )
            return {'status': 'success', 'response': response.choices[0].message.content, 'pii_detected': len(privacy_check.pii_found) > 0, 'risk_level': privacy_check.risk_level.value}
        except Exception as e:
            return {'status': 'error', 'error': str(e)}

assistant = PrivacyPreservingAssistant()
test_messages = [
    'Hello, how are you?',
    'My email is test@example.com',
    'What is the weather today?'
]
print('=== Privacy-Preserving Assistant ===\n')
for msg in test_messages:
    print(f'User: {msg}')
    result = assistant.process_message(msg)
    print(f'Status: {result["status"]}')
    print(f'PII Detected: {result.get("pii_detected", False)}')
    print()

## Failure Case: Privacy Protection Limitations

In [ ]:
limitations = [
    {'name': 'Contextual PII', 'issue': 'Information that becomes PII in context', 'example': 'The CEO of Acme Corp lives at this address'},
    {'name': 'Indirect Identification', 'issue': 'Combining non-PII to identify individuals', 'example': 'Male, 35, lives in ZIP 90210, works at Tech Co'},
    {'name': 'Model Memorization', 'issue': 'LLMs may remember training data PII', 'example': 'Email addresses from training corpus'},
    {'name': 'Inference Attacks', 'issue': 'Deducing PII from model outputs', 'example': 'Membership inference attacks'}
]
print('=== Privacy Protection Limitations ===\n')
for lim in limitations:
    print(f"⚠️ {lim['name']}")
    print(f"   Issue: {lim['issue']}")
    print(f"   Example: {lim['example']}")
    print()
print('Mitigation: Use differential privacy, federated learning, and regular audits.')

## Benchmark: Privacy Protection Methods

In [ ]:
import pandas as pd
methods = {
    'Method': ['Regex Patterns', 'ML NER', 'Presidio', 'Differential Privacy', 'Federated Learning', 'Homomorphic Encryption'],
    'Accuracy': ['Medium', 'High', 'Very High', 'N/A', 'N/A', 'N/A'],
    'Performance': ['Fast', 'Medium', 'Fast', 'Slow', 'Medium', 'Very Slow'],
    'Complexity': ['Low', 'Medium', 'Low', 'High', 'High', 'Very High'],
    'Use Case': ['Basic PII', 'Advanced NER', 'Enterprise', 'Statistical queries', 'Distributed training', 'Secure computation']
}
df = pd.DataFrame(methods)
print(df.to_string(index=False))

## Interactive Playground

In [ ]:
examples = [
    ('Safe', 'Hello, nice to meet you!'),
    ('Email', 'Contact me at user@domain.com'),
    ('Phone', 'Call me at 555-123-4567'),
    ('SSN', 'My SSN is 123-45-6789'),
    ('Credit Card', 'Card: 1234-5678-9012-3456')
]
print('=== Privacy Protection Demo ===\n')
for category, text in examples:
    result = protector.protect(text)
    icon = '🔒' if result.pii_found else '✅'
    print(f'{icon} [{category}] {text}')
    print(f'   Sanitized: {result.sanitized_text}')
    print(f'   Risk: {result.risk_level.value}')
    print()

## Tips & Tricks

- Always mask PII before sending to LLMs
- Use data minimization - collect only what's needed
- Implement audit logs for data access
- Regular privacy impact assessments
- Consider differential privacy for analytics

## References

1. **Microsoft Presidio** - https://microsoft.github.io/presidio/
2. **GDPR Guidelines** - https://gdpr.eu/
3. **NIST Privacy Framework** - https://www.nist.gov/privacy-framework